In [10]:
import numpy as np
import json
from torch.utils.data.dataset import TensorDataset
import  torch

In [15]:
with open("../activations/imagenet_train_hf/config.json") as f:
    config = json.load(f)


test = np.memmap(
    "../activations/imagenet_train_hf/embeddings.npy",
    dtype=np.float16,
    mode="r",
    shape=(config["dataset_size"], config["clip_embedding_size"]),
)
labels = np.memmap(
    "../activations/imagenet_train_hf/labels.npy",
    dtype=np.int64,
    mode="r",
    shape=(config["dataset_size"],),
)


dataset = TensorDataset(torch.from_numpy(test), torch.from_numpy(labels))


In [20]:
from torch.utils.data.dataloader import DataLoader

dl = DataLoader(dataset, batch_size=8)

for batch in dl:
    tensors, labels = batch

    print(tensors.shape)
    print(labels.shape)
    break

torch.Size([8, 512])
torch.Size([8])


In [23]:
from ca_sae.sae.batch_top_k import BatchTopKSAE

sae = BatchTopKSAE.from_pretrained(
    "../checkpoints/test/BatchTopKSAEFirst/ae.pt", k=64, device="cuda"
)

In [32]:
for batch in dl:
    x = batch[0].to("cuda")
    f = sae.encode(x)

    x_hat = sae.decode(f)

    loss = (x - x_hat)**2
    # loss = torch.nn.functional.mse_loss(x_hat, x)

    print(loss)
    break

tensor([[1.1086e-03, 5.2704e-03, 9.0683e-03,  ..., 3.9293e-03, 2.0399e-03,
         2.6667e-02],
        [8.2772e-04, 8.4545e-02, 2.2840e-02,  ..., 1.0566e-03, 2.6218e-04,
         7.7541e-05],
        [6.1778e-03, 1.4070e-02, 1.3940e-03,  ..., 1.0446e-03, 9.9758e-03,
         3.1786e-03],
        ...,
        [1.4884e-03, 9.6716e-03, 5.0570e-03,  ..., 5.5749e-04, 2.7411e-03,
         3.1027e-03],
        [4.0905e-02, 3.9162e-02, 4.5243e-02,  ..., 6.4301e-02, 3.0533e-02,
         7.3943e-03],
        [2.1302e-02, 2.1648e-02, 1.1654e-02,  ..., 2.8811e-02, 8.7137e-03,
         1.2025e-02]], device='cuda:0', grad_fn=<PowBackward0>)
